# Decoding an upcoming decision bias from pre-stimulus neural activity

**Dataset: DANDI:000409, the International Brain Laboratory Brain Wide Map**

Mice perform a two-alternative visual contrast-detection task: a Gabor patch
appears on the left or the right of a screen and the mouse reports its side by
turning a wheel. After the first ~90 trials of each session the side is no
longer drawn 50/50. It is drawn from hidden blocks of 20 to 100 trials in which
P(stimulus on left) is either 0.8 or 0.2, and the blocks alternate. Mice learn
to track this hidden prior, and it biases their choices when the stimulus is
weak or absent.

That prior is a *decision bias that exists before the stimulus appears*. The
question this notebook asks is whether it can be read out of population spiking
in the 400 ms preceding stimulus onset, on single trials, using held-out data.

The task makes this a clean test in two ways. The stimulus has not yet been
presented, so nothing about the current sensory evidence is available. And the
task enforces a quiescence period of 400 to 700 ms before stimulus onset during
which the wheel must be held still, so the analysis window is a genuinely
stationary baseline rather than a movement epoch.

## What is hard about this, and what the analysis does about it

Blocks are contiguous runs of trials, so block identity is strongly
autocorrelated in time. Anything else that drifts slowly over a session,
electrode motion in particular, is therefore partially confounded with the
label. Two things follow.

First, cross-validation has to hold out whole blocks, not random trials.
Second, the null distribution cannot be a trial shuffle, which destroys the
autocorrelation and makes almost anything look significant. We use
pseudo-sessions: surrogate block sequences built from the session's own block
lengths, in random order with a random starting phase, run through the identical
pipeline.

Even with those two in place, an unfiltered decoder latches onto slow drift and
then generalises *inversely* to held-out blocks (we measured balanced
accuracies as low as 0.04, far below the 0.5 chance level). Removing drift with
a moving-average high-pass along the trial axis fixes this. The filter width and
the ridge penalty are chosen leave-one-session-out, so no session contributes to
its own hyper-parameters.

## Setup

The helper modules that sit next to this notebook are:

* `ibl_io.py` — streaming access to DANDI:000409 through `remfile` with a local
  disk cache, reading only the datasets we need rather than whole NWB files.
* `analysis_lib.py` — trial construction, pre-stimulus spike counts via Pynapple,
  the drift high-pass, block-held-out cross-validation, the pseudo-session null,
  and the confound controls.
* `figures_lib.py` — figure construction.

In [1]:
import json
import os
import sys
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
os.chdir(HERE)
sys.path.insert(0, HERE)
warnings.filterwarnings("ignore")

import ibl_io as io
import analysis_lib as al
import figures_lib as F

print("pre-stimulus window:", al.PRE_WIN, "s relative to Gabor onset")

pre-stimulus window: (-0.4, 0.0) s relative to Gabor onset


## Session selection

DANDI:000409 holds 459 processed `behavior+ecephys` NWB files. We scanned a
random sample of 90 of them (`02_scan_sessions.py`) for trial count, block
structure, unit yield and anatomical coverage, then took 13 sessions from 13
different mice that span frontal cortex, striatum, thalamus, midbrain,
hippocampus, visual cortex and amygdala. Selection used only recording quality
and anatomy, never anything about the effect being tested.

In [2]:
selected = json.load(open("selected_sessions.json"))
print(f"{len(selected)} sessions from {len(set(s['subject'] for s in selected))} mice")
pd.DataFrame([{"subject": s["subject"], "trials": s["n_trials"],
               "biased trials": s["n_biased"], "GB": round(s["size"] / 1e9, 2),
               "regions": {k: v for k, v in s["counts"].items() if v > 0}}
              for s in selected])

13 sessions from 13 mice


,subject,trials,biased trials,GB,regions
0,ZM-1897,413,323,0.62,"{'frontal': 600, 'thalamus': 122, 'hippocampus..."
1,SWC-038,767,677,1.11,"{'frontal': 625, 'other': 247}"
2,CSH-ZAD-019,672,582,1.49,"{'frontal': 143, 'thalamus': 279, 'hippocampus..."
3,NR-0031,489,399,1.08,"{'thalamus': 4, 'hippocampus': 460, 'visual': ..."
4,CSH-ZAD-026,1103,1013,1.45,"{'frontal': 323, 'striatum': 154, 'other': 328}"
5,SWC-043,551,461,1.38,"{'frontal': 11, 'striatum': 186, 'thalamus': 2..."
6,ZM-2241,872,782,1.63,"{'frontal': 190, 'striatum': 205, 'other': 513}"
7,UCLA037,545,455,0.48,"{'hippocampus': 262, 'visual': 111}"
8,ZFM-01935,743,653,1.85,"{'thalamus': 270, 'midbrain': 12, 'hippocampus..."
9,NR-0020,817,727,1.48,"{'midbrain': 205, 'hippocampus': 124, 'other':..."


## Loading a session

Files are streamed, never downloaded whole. `pynwb`'s `units.to_dataframe()`
eagerly materialises `waveform_mean` and `spike_amplitudes`, which dominate file
size, so `ibl_io` reads the few datasets we need directly through `h5py` over
`remfile`. Spike times become a Pynapple `TsGroup` and every spike count in this
notebook is computed with `TsGroup.count` over Pynapple `IntervalSet`s.

In [3]:
assets = {a[0]: a for a in io.list_processed_assets()}
demo = selected[0]
hf = io.open_h5(assets[demo["path"]][2])
trials_raw = al.build_trials(hf)
print(demo["subject"], "raw trials:", len(trials_raw))
trials_raw[["stim_on", "gabor_stimulus_contrast", "gabor_stimulus_side",
            "mouse_wheel_choice", "probability_left", "block_id",
            "quiescence_period"]].head(8)

ZM-1897 raw trials: 413


,stim_on,gabor_stimulus_contrast,gabor_stimulus_side,mouse_wheel_choice,probability_left,block_id,quiescence_period
0,8.904133,100.00,right,counter_clockwise,0.5,0,0.541949
1,11.837167,25.00,left,clockwise,0.5,0,0.457997
2,14.687133,25.00,right,counter_clockwise,0.5,0,0.426862
3,18.203467,12.50,right,counter_clockwise,0.5,0,0.423191
4,24.653433,25.00,right,counter_clockwise,0.5,0,0.501897
5,27.819500,100.00,right,counter_clockwise,0.5,0,0.566746
6,31.269367,6.25,left,counter_clockwise,0.5,0,0.487259
7,35.585600,100.00,left,clockwise,0.5,0,0.646090


### Verifying the wheel-turn convention

The NWB file stores the choice as a wheel direction rather than a reported side.
We map clockwise to "reported left" and check the mapping against
high-contrast trials, where the mouse is almost always correct. If the mapping
were inverted this number would come out near zero.

In [4]:
conv, n_easy = al.check_choice_convention(trials_raw)
print(f"{conv*100:.1f}% correct on {n_easy} trials at >=50% contrast")
assert conv > 0.8, "wheel-turn convention is inverted"

98.8% correct on 85 trials at >=50% contrast


### Trial inclusion

A trial enters the analysis if the mouse made a choice, the trial is in a biased
block (the unbiased warm-up trials are dropped), the 400 ms window fits inside
the trial, and no wheel movement was detected inside the window.

In [5]:
ok = al.valid_trials(trials_raw)
print(f"{ok.sum()} of {len(trials_raw)} trials usable "
      f"({(~ok).sum()} dropped: unbiased block, no choice, or movement in window)")

296 of 413 trials usable (117 dropped: unbiased block, no choice, or movement in window)


## Extraction

`05_extract.py` streams each session once and caches the trial table, the
pre-stimulus spike-count matrix, sliding-window count matrices from -1.0 s to
+0.6 s, peri-stimulus spike times, and the pre-stimulus wheel regressor.
`05b_patch_behaviour.py` then adds the per-unit quality metrics and the pupil
and face motion-energy regressors for the sessions that carry video. Everything
downstream reads those caches, so the notebook re-runs in seconds once the
cache exists.

Units enter the analysis if they pass at least two of the three IBL isolation
metrics, fire at least 0.5 Hz, sit outside white matter, and are present
throughout the session (presence ratio >= 0.9). That last criterion matters:
a unit that switches off partway through has counts the drift high-pass cannot
repair, and such units otherwise dominate the extreme block AUCs.

In [6]:
if len(os.listdir("cache")) < len(selected):
    assert os.system(f"{sys.executable} 05_extract.py") == 0
assert os.system(f"{sys.executable} 05b_patch_behaviour.py") == 0
print(sorted(os.listdir("cache")))

patching:   0%|          | 0/13 [00:00<?, ?it/s]

patching:   8%|▊         | 1/13 [00:00<00:07,  1.70it/s]

patching:  15%|█▌        | 2/13 [00:01<00:08,  1.31it/s]

patching:  31%|███       | 4/13 [00:01<00:03,  2.45it/s]

patching:  38%|███▊      | 5/13 [00:02<00:03,  2.54it/s]

...

patching:  85%|████████▍ | 11/13 [00:06<00:01,  1.68it/s]

ZM-1897 {'wheel_speed': 296}


patching:  92%|█████████▏| 12/13 [00:07<00:00,  1.22it/s]

patching: 100%|██████████| 13/13 [00:08<00:00,  1.52it/s]


['CSH-ZAD-019_5adab0b7.pkl', 'CSH-ZAD-026_81a78eac.pkl', 'CSHL059_37e96d0b.pkl', 'DY-014_b39752db.pkl', 'KS014_16693458.pkl', 'NR-0020_eacc49a9.pkl', 'NR-0031_642c97ea.pkl', 'SWC-038_4d8c7767.pkl', 'SWC-043_6f09ba7e.pkl', 'UCLA037_bda2faf5.pkl', 'ZFM-01935_1a507308.pkl', 'ZM-1897_dd4da095.pkl', 'ZM-2241_ff4187b5.pkl']


## Decoding

`06_decode.py` runs the whole statistical pipeline:

1. a grid over high-pass width `w` and ridge penalty `C`, with the true labels;
2. leave-one-session-out selection of `(w, C)` for each session;
3. the observed accuracy and a 500-sample pseudo-session null;
4. controls: a previous-trial behavioural-history decoder, a stacked comparison
   of the two, and a repeat on history-matched trials;
5. sliding-window decoding with its own 200-sample null per window;
6. per-unit AUCs with a 200-sample null;
7. decoding of the upcoming *choice* on weak-stimulus trials.

In [7]:
if not os.path.exists("results.pkl"):
    assert os.system(f"{sys.executable} 06_decode.py") == 0
blob = pd.read_pickle("results.pkl")
R, G, WS, CS, params = (blob["results"], blob["grid"], blob["WS"], blob["CS"],
                        blob["params"])
print(f"{len(R)} sessions | {sum(r['n_units'] for r in R)} units | "
      f"{sum(r['n_trials'] for r in R)} trials")

13 sessions | 7104 units | 6670 trials


## The behaviour we are trying to decode

Before looking at spikes, confirm the bias exists. Psychometric curves shift
horizontally between the two block types, the shift is present in essentially
every session at the weakest contrasts, and it builds up over the first several
trials after a block switch rather than appearing instantly.

In [8]:
bias = F.fig_behavior(R)
print(f"bias at <=6.25% contrast: median {np.median(bias):+.3f}, "
      f"Wilcoxon p={stats.wilcoxon(bias)[1]:.2g}")

wrote fig01_task_and_behavior.png
bias at <=6.25% contrast: median +0.362, Wilcoxon p=0.00024


![](fig01_task_and_behavior.png)

## Raw data

Raw spiking with the pre-stimulus windows marked, the most block-selective unit
in the session as a raster and a block-split PSTH, the wheel trace confirming
quiescence, and the drift-removed pre-stimulus population matrix sorted by
block.

In [9]:
F.fig_raw(R, which=int(np.argmax([r["acc"] for r in R])))

wrote fig02_raw_data.png


![](fig02_raw_data.png)

## Single units

Each unit's pre-stimulus spike count is scored by the area under the ROC curve
for discriminating the two block types, with significance from the same
pseudo-session null. Individual effects are small, as expected for a slow
internal variable, but the fraction of modulated units is well above 5%.

In [10]:
F.fig_single_units(R)

wrote fig03_single_unit_selectivity.png


![](fig03_single_unit_selectivity.png)

## Population decoding: the main result

In [11]:
accs, z, p_pooled = F.fig_decoding(R)
for r in sorted(R, key=lambda x: -x["acc"]):
    print(f"{r['subject']:>14s}  w={str(r['w']):>4s} C={r['C']:<6g} "
          f"acc={r['acc']:.3f}  null={r['null'].mean():.3f}  p={r['p']:.4f}")
print(f"\nmean accuracy {accs.mean():.3f}, pooled pseudo-session p={p_pooled:.2g}, "
      f"median z={np.median(z):.2f}, "
      f"Wilcoxon across sessions p={stats.wilcoxon(z)[1]:.2g}")

wrote fig04_block_decoding.png
       ZM-1897  w=  81 C=0.01   acc=0.706  null=0.494  p=0.0020
       CSHL059  w=  81 C=0.01   acc=0.664  null=0.505  p=0.0040
       ZM-2241  w=  81 C=0.01   acc=0.662  null=0.509  p=0.0020
       SWC-043  w=  81 C=0.01   acc=0.661  null=0.491  p=0.0020
   CSH-ZAD-019  w=  81 C=0.01   acc=0.650  null=0.478  p=0.0020
   CSH-ZAD-026  w=  81 C=0.01   acc=0.634  null=0.494  p=0.0020
       NR-0020  w=  81 C=0.01   acc=0.631  null=0.473  p=0.0020
       NR-0031  w=  81 C=0.01   acc=0.626  null=0.517  p=0.0200
     ZFM-01935  w=  81 C=0.01   acc=0.618  null=0.478  p=0.0040
        DY-014  w=  81 C=0.01   acc=0.611  null=0.484  p=0.0020
         KS014  w=  81 C=0.01   acc=0.607  null=0.529  p=0.0659
       UCLA037  w=  81 C=0.01   acc=0.573  null=0.479  p=0.0220
       SWC-038  w=  81 C=0.01   acc=0.568  null=0.518  p=0.0739

mean accuracy 0.632, pooled pseudo-session p=0.002, median z=2.88, Wilcoxon across sessions p=0.00024


![](fig04_block_decoding.png)

## Time-resolved decoding

Sliding 200 ms windows from 1 s before to 0.6 s after stimulus onset. The signal
is present across the whole pre-stimulus period, not just immediately before
onset, which is what a slowly varying prior should look like.

In [12]:
centers, tr_mean, tr_p = F.fig_time_resolved(R)
for c, m, p in zip(centers, tr_mean, tr_p):
    print(f"  centre {c:+.2f} s  acc={m:.3f}  p={p:.3g}")

wrote fig05_time_resolved.png
  centre -1.00 s  acc=0.624  p=0.00498
  centre -0.90 s  acc=0.621  p=0.00498
  centre -0.80 s  acc=0.622  p=0.00498
  centre -0.70 s  acc=0.622  p=0.00498
  centre -0.60 s  acc=0.610  p=0.00498
  centre -0.50 s  acc=0.605  p=0.00498
  centre -0.40 s  acc=0.610  p=0.00498
  centre -0.30 s  acc=0.615  p=0.00498
  centre -0.20 s  acc=0.623  p=0.00498
  centre -0.10 s  acc=0.620  p=0.00498
  centre -0.00 s  acc=0.643  p=0.00498
  centre +0.10 s  acc=0.692  p=0.00498
  centre +0.20 s  acc=0.706  p=0.00498
  centre +0.30 s  acc=0.708  p=0.00498
  centre +0.40 s  acc=0.705  p=0.00498
  centre +0.50 s  acc=0.688  p=0.00498
  centre +0.60 s  acc=0.688  p=0.00498


![](fig05_time_resolved.png)

## Controls

The mouse can only know the block from recent trials, so the neural signal must
ultimately derive from trial history. The question a control can answer is
whether the pre-stimulus population state carries block information that is not
simply a trace of the immediately preceding trial. Two tests bear on this:
repeating the decode on trials matched for the previous trial's choice, reward
and stimulus side, and stacking the out-of-fold neural and history predictions
against each other.

The panel also shows the hyper-parameter grid, which is where the drift problem
is most visible: the bottom row is the no-filter condition, and it falls well
below chance.

In [13]:
F.fig_controls(R, G, WS, CS, params)
V = np.array([[r["ctrl"][k] for k in ["history", "neural", "both"]] for r in R])
print("history-only  %.3f | neural-only %.3f | both %.3f (session means)"
      % tuple(V.mean(0)))
beta = np.array([r["stack"]["beta_neural"] for r in R])
print(f"stacked neural coefficient given history: median {np.median(beta):+.3f}, "
      f"positive in {int((beta>0).sum())}/{len(beta)} sessions, "
      f"Wilcoxon p={stats.wilcoxon(beta)[1]:.2g}")
pm = np.array([r["p_matched"] for r in R], float)
am = np.array([r["acc_matched"] for r in R], float)
print(f"history-matched trials: mean accuracy {np.nanmean(am):.3f}, "
      f"{int(np.nansum(pm < 0.05))}/{int(np.isfinite(pm).sum())} sessions p<0.05")

wrote fig06_controls.png
history-only  0.783 | neural-only 0.630 | both 0.718 (session means)
stacked neural coefficient given history: median +0.431, positive in 13/13 sessions, Wilcoxon p=0.00024
history-matched trials: mean accuracy 0.621, 10/13 sessions p<0.05


![](fig06_controls.png)

## Decoding the upcoming choice

The block prior is the experimenter's variable. The behaviourally meaningful
one is the choice the mouse is about to make. On trials where the stimulus is
weak (contrast <= 12.5%) the choice is largely driven by the prior, so if the
prior is present before onset the upcoming choice should be partially
predictable from the same window. The third panel asks whether the decoded
prior predicts choice *over and above* the true block label, and the second
whether choice is still predicted within a single block.

In [14]:
ch_acc, ch_within, ch_coefs = F.fig_choice(R)
print(f"choice decoding: mean {ch_acc.mean():.3f}, "
      f"Wilcoxon vs 0.5 p={stats.wilcoxon(ch_acc - 0.5)[1]:.3g}")
print(f"within-block:    mean {ch_within.mean():.3f}, "
      f"Wilcoxon vs 0.5 p={stats.wilcoxon(ch_within - 0.5)[1]:.3g}")
print(f"decoded-prior weight on choice given block label: "
      f"median {np.median(ch_coefs[:, 0]):+.3f}, "
      f"p={stats.wilcoxon(ch_coefs[:, 0])[1]:.3g}")

wrote fig07_choice_decoding.png
choice decoding: mean 0.535, Wilcoxon vs 0.5 p=0.000488
within-block:    mean 0.512, Wilcoxon vs 0.5 p=0.455
decoded-prior weight on choice given block label: median +0.092, p=0.0327


![](fig07_choice_decoding.png)

## Anatomy

Units are grouped by the Allen region of their peak channel. This is a coverage
summary rather than a controlled comparison: the 13 probes were not placed to
sample regions evenly, and per-region unit counts differ by an order of
magnitude.

In [15]:
region_table = F.fig_regions(R)
print(region_table.round(3).to_string())

wrote fig08_regions.png
                          n  nses   frac    dev
group                                          
frontal cortex         1033     6  0.119  0.041
thalamus               1535     6  0.093  0.032
other                  1569    12  0.086  0.033
midbrain                620     4  0.084  0.034
sensory cortex          439     7  0.077  0.036
striatum / pallidum     639     5  0.075  0.030
hippocampal formation  1053     9  0.052  0.033
amygdala                216     3  0.046  0.029


![](fig08_regions.png)

## Summary

`08_write_readme.py` recomputes every statistic quoted in `README.md` from
`results.pkl` and writes `summary.json` and `per_session_results.csv`, so the
prose and the analysis cannot drift apart.

In [16]:
assert os.system(f"{sys.executable} 08_write_readme.py > /dev/null") == 0
summary = json.load(open("summary.json"))
for k, v in summary.items():
    print(f"{k:>28s}: {v}")

                    dandiset: DANDI:000409
                    sessions: 13
                        mice: 13
                       units: 7104
units_before_presence_filter: 9287
                      trials: 6670
                pre_window_s: [-0.4, 0.0]
           behav_bias_median: 0.3620370370370371
                behav_bias_p: 0.000244140625
               mean_accuracy: 0.6315710992773755
                   mean_null: 0.49602110977119634
                sessions_p05: 11
                    pooled_p: 0.001996007984031936
                    median_z: 2.8768929180479597
            across_session_p: 0.000244140625
        frac_units_modulated: 0.0843186936936937
               units_binom_p: 3.951370490574407e-34
       matched_mean_accuracy: 0.6208649734872496
        matched_sessions_p05: 10
                   matched_n: 13
            stack_acc_neural: 0.6302687245500173
           stack_acc_history: 0.7828495920851748
              stack_acc_both: 0.7841352704209668
         s